# Text classification

Using the dataset `dataset_emails.csv` (or other dataset of your choice) create three text classificators:
* Using rule-based approach (regex)
* Using naive-bayes
* Using Spacy 3 

Finally, compare the results and show what is better and why. 

A classifier is to label the documents. 

In [10]:
import re
import pandas as pd

In [11]:
# Load the dataset, each email of the dataset have a prompt that is the email text and the label that is the type of the email
file_path = "/Users/bernardoquindimil/Code/Berniquindimil/NLP_Digital_Portfolio/S08/dataset_emails.csv"
df = pd.read_csv(file_path)

In [ ]:
# Define regex patterns for each category
regex_rules = {
    "send": r"\b(send|write|draft|email someone|new message|new email)\b",
    "list": r"\b(list|show|view|all emails|inbox|check mailbox)\b",
    "trash": r"\b(delete|remove|trash|discard|bin)\b",
    "read": r"\b(read|open email|check email|view message|see email)\b",
    "reply": r"\b(reply|respond|answer email|write back)\b",
    "untrash": r"\b(restore|untrash|recover email|move from trash)\b",
    "forward": r"\b(forward|send again|pass along)\b",
    "star": r"\b(star|important|mark as important|highlight)\b",
    "trash_list": r"\b(view trashed|show deleted|list trash)\b",
}

# Function to classify emails using regex
def classify_email(prompt):
    for label, pattern in regex_rules.items():
        if re.search(pattern, prompt, re.IGNORECASE):
            return label
    return "unknown"  # Default category if no pattern matches

# Apply classification adding other column to the dataset with the predicted label 
df["predicted_label"] = df["prompt"].apply(classify_email)

# Evaluate results by checking mismatches
mismatches = df[df["label"] != df["predicted_label"]]

# Display a sample of mismatches
mismatches.head(10)


                                      prompt label predicted_label
0               Can I send an email, please?  send            send
1              I'd like to compose an email.  send         unknown
2                   I need to send an email.  send            send
3          Could you help me write an email?  send            send
4  Is it possible to send an email with you?  send            send


,prompt,label,predicted_label
1,I'd like to compose an email.,send,unknown
8,Open email for writing.,send,read
11,There's someone I need to email.,send,unknown
12,I want to get in touch with [someone] through ...,send,unknown
16,Let's shoot someone an email.,send,unknown
18,Is there a way to email [someone]?,send,unknown
20,I need to drop someone a line.,send,unknown
21,Let's ping someone with an email.,send,unknown
22,Time to fire off an email.,send,unknown
24,Can you whip up an email for me?,send,unknown


This isn't accurate because there are a lot emails and classify with concrete regex patterns seeing all the emails is very difficult because there are 1000 without concrete patterns and structures depending of the type. Besides, concrete regex pattern could be used in different emails. For example, draft, appears 24 times in the document in differents types of emails (send, list, reply). Although I put some patterns that are relationated with some emails of each type, there are 10 dismatches in the first 26 emails. And for example, I put the pattern 'email someone' in type send but there are send emails that appear "someone I need to email" and I can't put all the possible patterns and are better ways more efficient with automatization.

In the rule-based approach is better to apply technics like lemmatization and tokenization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
file_path = "/Users/bernardoquindimil/Code/Berniquindimil/NLP_Digital_Portfolio/S08/dataset_emails.csv"
df = pd.read_csv(file_path)

# Basic text preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\W+", " ", text)  # Remove special characters \W is a char that isn't a lettre, digit or "_"
    return text

df["clean_prompt"] = df["prompt"].apply(clean_text)

# Convert text into numerical features using TF-IDF
# It quantify the importance of a given word relative to other words in the document and in the corpus for reducing importance to the stopwords
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["clean_prompt"])  # Features
y = df["label"]  # Target labels

# Split data into training and test sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Naïve Bayes model
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# Predictions
y_pred = nb_model.predict(X_test)

# Evaluate performance
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}\n")
print("Classification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.7750

Classification Report:
               precision    recall  f1-score   support

     forward       0.96      0.92      0.94        25
        list       0.77      0.77      0.77        13
        read       0.79      0.71      0.75        21
       reply       0.56      0.93      0.70        15
        send       0.83      0.75      0.79        20
        star       0.54      1.00      0.70        13
       trash       0.94      0.59      0.73        27
  trash_list       0.83      0.87      0.85        23
     unknown       0.80      0.38      0.52        21
     untrash       0.81      0.95      0.88        22

    accuracy                           0.78       200
   macro avg       0.78      0.79      0.76       200
weighted avg       0.81      0.78      0.77       200



The accuracy is very good but it can be better. Forward, Untrash and trush list f1-scores are very accuracy. There are a lot of emails are bad classify as unknown.